In [4]:
import pandas as pd
import numpy as np
import os

In [5]:
RANDOM_STATE = 42

In [6]:
df = pd.read_csv('../data/raw/housing.csv')
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [7]:
# Leave the missing total_bedroom values for now, we could use a simple imputer to fill them in later.
# Alternatively, we could drop them. 207/20640 = 1% of the data, so we can afford to lose some of it without losing too much information.

In [8]:
# Use NaN-safe denominator to avoid division-by-zero issues
households_safe = df['households'].replace(0, np.nan)
rooms_safe = df['total_rooms'].replace(0, np.nan)
population_safe = df['population'].replace(0, np.nan)

# Ratio-based features
df['rooms_per_household'] = df['total_rooms'] / households_safe
df['bedrooms_per_room'] = df['total_bedrooms'] / rooms_safe
df['bedrooms_per_household'] = df['total_bedrooms'] / households_safe
df['population_per_household'] = df['population'] / households_safe
df['rooms_per_person'] = df['total_rooms'] / population_safe

# Income-density features
df['income_per_household_member'] = df['median_income'] / df['population_per_household']
df['income_x_rooms_per_household'] = df['median_income'] * df['rooms_per_household']

# Geographic interaction features
df['lat_x_lon'] = df['latitude'] * df['longitude']
df['lat_plus_lon'] = df['latitude'] + df['longitude']

# Log-transform skewed count features
for col in ['total_rooms', 'total_bedrooms', 'population', 'households']:
    df[f'log_{col}'] = np.log1p(df[col])

df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,...,population_per_household,rooms_per_person,income_per_household_member,income_x_rooms_per_household,lat_x_lon,lat_plus_lon,log_total_rooms,log_total_bedrooms,log_population,log_households
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,...,2.555556,2.732919,3.257687,58.144254,-4630.0724,-84.35,6.781058,4.867534,5.777652,4.844187
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,...,2.109842,2.956685,3.934608,51.785271,-4627.2492,-84.36,8.867850,7.009409,7.784057,7.037906
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,...,2.802260,2.957661,2.589838,60.150315,-4626.7840,-84.39,7.291656,5.252273,6.208590,5.181784
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,...,2.547945,2.283154,2.214765,32.827897,-4627.1625,-84.40,7.150701,5.463832,6.326149,5.393628
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,...,2.181467,2.879646,1.763125,24.161264,-4627.1625,-84.40,7.395108,5.638355,6.338594,5.560682
